## Setup — run this cell first
Makes sure Python imports from the **project folder**, not from any
other `pipeline` package that may be installed in your environment.


In [2]:
import sys, os

PROJECT_DIR = r'C:\Users\Zakaria\Downloads\digital_project\old-photo-restoration-dip'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


Working directory: C:\Users\Zakaria\Downloads\digital_project\old-photo-restoration-dip


# Old Photo Restoration
## Digital Image Processing — Final Pipeline

---

### What this notebook does
Old photographs suffer from several types of damage:
- **Noise and grain** — analog film grain and scanner noise
- **Scratches and cracks** — thin bright lines from physical damage
- **Low contrast** — fading caused by aging paper
- **Sepia / color cast** — yellowing over decades

### Pipeline
```
Original Image
     |
     v
Step 1 -- Denoising          (NLM removes grain)
     |
     v
Step 2 -- Scratch Detection  (4-direction top-hat -> binary mask)
     |
     v
Step 3 -- Inpainting         (Navier-Stokes + post-blend fills damage)
     |
     v
Step 4 -- Contrast (CLAHE)   (improve local contrast)
     |
     v
Step 5 -- Color Correction   (fix sepia / grayscale)
     |
     v
Final Restored Image
```

> All logic lives in `pipeline/`. This notebook calls the modules
> directly — no duplicated code.


---
## 1. Imports and Helper Function


In [3]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from pipeline import ScratchDetector, Denoiser, Inpainter, Enhancer, Colorizer


def show(images, titles, cmaps=None, figsize=None):
    """Display a row of images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize or (5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, img, title, cmap in zip(axes, images, titles, cmaps or [None]*n):
        if cmap:
            ax.imshow(img, cmap=cmap)
        elif img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


ImportError: cannot import name 'ScratchDetector' from 'pipeline' (c:\Users\Zakaria\anaconda3\Lib\site-packages\pipeline\__init__.py)

---
## 2. Load Images


In [ ]:
IMAGE_PATHS = sorted(
    glob.glob(os.path.join("assets", "gallery_originals", "*.jpg")) +
    glob.glob(os.path.join("assets", "gallery_originals", "*.jpeg")) +
    glob.glob(os.path.join("assets", "gallery_originals", "*.png"))
)

originals = []
for path in IMAGE_PATHS:
    img = cv2.imread(path)
    if img is None:
        print(f"WARNING: could not load {path}")
    else:
        originals.append(img)
        print(f"Loaded {os.path.basename(path)} -- {img.shape[1]}x{img.shape[0]}")

print(f"\nTotal: {len(originals)} images")
show(originals, [f"Original {i+1}" for i in range(len(originals))])


---
## Step 1 -- Denoising

Noise must be removed **before** scratch detection — otherwise random grain
gets mistaken for damage.

**Non-Local Means (NLM)** finds similar patches across the whole image and
averages them. It preserves edges and skin texture better than a median filter.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `h` | 10 | filter strength (5=mild, 15=strong) |
| `templateWindowSize` | 7 | patch size for comparison |
| `searchWindowSize` | 21 | search radius for similar patches |


In [ ]:
NLM_H        = 10
NLM_TEMPLATE = 7
NLM_SEARCH   = 21

denoiser = Denoiser(nlm_h=NLM_H, nlm_template=NLM_TEMPLATE, nlm_search=NLM_SEARCH)

denoised_all = []
for i, img in enumerate(originals):
    denoised = denoiser.nlm_filter(img)
    denoised_all.append(denoised)
    print(f"Image {i+1}")
    show([img, denoised], ["Original", f"Denoised (NLM h={NLM_H})"])

print("Denoising complete.")


### Tuning: compare h = 5, 10, 15
Run on image 1 to decide the best NLM strength.


In [ ]:
img_test = originals[0]
show(
    [img_test,
     cv2.fastNlMeansDenoisingColored(img_test, None,  5,  5, 7, 21),
     cv2.fastNlMeansDenoisingColored(img_test, None, 10, 10, 7, 21),
     cv2.fastNlMeansDenoisingColored(img_test, None, 15, 15, 7, 21)],
    ["Original", "h=5 (mild)", "h=10 (default)", "h=15 (strong)"]
)
# h=10 is the best balance: grain removed, facial texture kept.


---
## Step 2 -- Scratch Detection

Output: a **binary mask** (white = damaged, black = clean).

**Method: Morphological Top-Hat** at 4 orientations (H, V, diagonal, anti-diagonal).
The top-hat highlights features that are brighter than their surroundings
and narrower than the structuring element — exactly what scratches look like.

Key protections:
- **Aspect-ratio filter**: keeps only elongated components (min 2:1)
- **Density filter**: rejects blobs (area/bbox must be low for lines)
- **Safety cap**: if mask > 8% of image it is discarded (prevents destroying photo)

| Parameter | Value | Notes |
|-----------|-------|-------|
| `kernel_length` | 23 | line detector length |
| `bright_cutoff` | 25 | ignore pixels darker than this |
| `adaptive_factor` | 0.50 | fraction of 97th percentile used as threshold |
| `min_scratch_len` | 10 | minimum component size to keep |
| `min_aspect_ratio` | 2.0 | length/width ratio to count as a line |
| `density_threshold` | 0.20 | area/bbox must be below this for lines |
| `max_mask_ratio` | 0.08 | safety cap: reject mask if it covers > 8% of pixels |
| `dilation_iter` | 1 | expand mask to cover full scratch width |


In [ ]:
detector = ScratchDetector(
    kernel_length=23,
    bright_cutoff=25,
    adaptive_factor=0.50,
    min_scratch_len=10,
    min_aspect_ratio=2.0,
    dilation_iter=1,
    max_mask_ratio=0.08,
    max_component_area_ratio=0.015,
    min_threshold=25,
    density_threshold=0.20,
    verbose=True,
)

masks_all = []
for i, img in enumerate(denoised_all):
    mask, debug = detector.detect(img, return_debug=True)
    masks_all.append(mask)

    overlay = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    overlay[mask == 255] = [220, 50, 50]

    print(f"Image {i+1}: {debug[\"scratch_pixels\"]} px "
          f"({100*debug[\"scratch_ratio\"]:.1f}%) | "
          f"threshold={debug[\"adaptive_threshold\"]}")
    show([img, mask, overlay],
         ["Denoised", "Scratch Mask", "Scratches (red)"],
         cmaps=[None, "gray", None])

print("Scratch detection complete.")


### Tuning: compare adaptive_factor = 0.4, 0.50, 0.7


In [ ]:
img_test = denoised_all[0]
gray     = cv2.cvtColor(img_test, cv2.COLOR_BGR2GRAY)

h_k  = cv2.getStructuringElement(cv2.MORPH_RECT, (23, 1))
v_k  = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 23))
d1_k = np.zeros((23, 23), dtype=np.uint8)
np.fill_diagonal(d1_k, 1)
d2_k = np.zeros((23, 23), dtype=np.uint8)
np.fill_diagonal(np.fliplr(d2_k), 1)

combined = np.zeros_like(gray)
for k in [h_k, v_k, d1_k, d2_k]:
    combined = np.maximum(combined, cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k))

p97 = np.percentile(combined, 97)
results, labels = [], []
for fac in [0.4, 0.50, 0.7]:
    t = max(25, int(p97 * fac))
    _, m = cv2.threshold(combined, t, 255, cv2.THRESH_BINARY)
    m = cv2.dilate(m, np.ones((3,3), np.uint8), iterations=1)
    pct = 100*cv2.countNonZero(m)/(gray.shape[0]*gray.shape[1])
    results.append(m)
    labels.append(f"factor={fac} (t={t}) | {pct:.1f}%")

show(results, labels, cmaps=["gray"]*3)
# 0.50 is the best balance for this dataset.


---
## Step 3 -- Inpainting

Fills in the masked pixels using the surrounding context.

**Navier-Stokes (NS)** is chosen over TELEA because it handles wider
and more complex damage better by continuing image isophotes smoothly.

After NS, a **light NLM pass** (h=5) blends the repaired seams
with the surrounding texture so the fill is invisible.

| Parameter | Value | Notes |
|-----------|-------|-------|
| `radius` | 7 | NS neighborhood radius |
| `blend_h` | 5 | NLM strength for post-blend (keep low: 3-5) |


In [ ]:
INPAINT_RADIUS = 7
BLEND_H        = 5

inpainted_all = []
for i, (img, mask) in enumerate(zip(denoised_all, masks_all)):
    inpainted = Inpainter(img, mask).navier_stokes_blend(
        radius=INPAINT_RADIUS, blend_h=BLEND_H
    )
    inpainted_all.append(inpainted)
    print(f"Image {i+1}")
    show([img, inpainted], ["Before Inpainting", "After NS + Blend"])

print("Inpainting complete.")


### Tuning: compare TELEA vs NS vs NS+blend on image 1


In [ ]:
img_t = denoised_all[0]
msk_t = masks_all[0]
r     = cv2.inpaint(img_t, msk_t, 3, cv2.INPAINT_TELEA)
ns    = cv2.inpaint(img_t, msk_t, 7, cv2.INPAINT_NS)
blend = cv2.fastNlMeansDenoisingColored(ns, None, 5, 5, 7, 21)

show([img_t, r, ns, blend],
     ["Denoised", "TELEA r=3", "NS r=7", "NS + blend h=5"])
# NS+blend gives the smoothest seam transition.


---
## Step 4 -- Contrast Enhancement (CLAHE)

Old photos often look flat or faded. CLAHE brightens dark regions locally
without blowing out bright ones.

We apply CLAHE only to the **L channel** in LAB color space so colors
stay the same — only brightness is adjusted.

| Parameter | Value | Notes |
|-----------|-------|-------|
| `clip_limit` | 2.0 | contrast cap per tile (1=subtle, 4=aggressive) |
| `tile_grid_size` | (8, 8) | local region size |


In [ ]:
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)

enhanced_all = []
for i, img in enumerate(inpainted_all):
    enhanced = Enhancer(img).apply_clahe(clip_limit=CLAHE_CLIP,
                                         tile_grid_size=CLAHE_TILE)
    enhanced_all.append(enhanced)
    print(f"Image {i+1}")
    show([img, enhanced], ["Before CLAHE", f"After CLAHE (clip={CLAHE_CLIP})"])

print("Contrast enhancement complete.")


### Tuning: compare clip = 1.0, 2.0, 4.0


In [ ]:
img_t = inpainted_all[0]
lab   = cv2.cvtColor(img_t, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)

results = []
for clip in [1.0, 2.0, 4.0]:
    lc = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8)).apply(l)
    results.append(cv2.cvtColor(cv2.merge((lc, a, b)), cv2.COLOR_LAB2BGR))

show([img_t]+results,
     ["Original", "clip=1.0 (subtle)", "clip=2.0 (balanced)", "clip=4.0 (strong)"])
# clip=2.0 is the best balance for this photo set.


---
## Step 5 -- Color Correction

Classification uses mean HSV saturation and hue:

| Type | Condition |
|------|-----------|
| grayscale | mean saturation < 15 |
| sepia | saturation 15-60 AND hue 10-30 (yellow-orange) |
| color | anything else |

**Sepia** -> gray-world white balance (equalize channel means).

**Grayscale** -> unsharp mask (alpha=2.5, beta=-1.5) + histogram stretch.


In [ ]:
colorizer = Colorizer(
    sepia_sat_min=15,
    sepia_sat_max=60,
    sepia_hue_min=10,
    sepia_hue_max=30,
)

final_all = []
for i, img in enumerate(enhanced_all):
    corrected, kind = colorizer.apply(img)
    final_all.append(corrected)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    print(f"Image {i+1}: detected as \"{kind}\" "
          f"(mean_sat={hsv[:,:,1].mean():.1f}, "
          f"mean_hue={hsv[:,:,0].mean():.1f})")
    show([img, corrected],
         [f"Before ({kind})", "After Color Correction"])

print("Color correction complete.")


---
## Step 6 -- Final Comparison
Side-by-side before / after for every image.


In [ ]:
for i, (orig, final) in enumerate(zip(originals, final_all)):
    print(f"-- Image {i+1} --")
    show([orig, final],
         [f"Original {i+1}", f"Restored {i+1}"],
         figsize=(12, 6))


---
## Step 7 -- Full Pipeline Progression
Every stage for every image in one view.


In [ ]:
STEP_LABELS = ["1. Original", "2. Denoised", "3. Mask",
               "4. Inpainted", "5. CLAHE", "6. Final"]

for i in range(len(originals)):
    steps = [originals[i], denoised_all[i], masks_all[i],
             inpainted_all[i], enhanced_all[i], final_all[i]]
    cmaps = [None, None, "gray", None, None, None]

    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    fig.suptitle(f"Image {i+1} -- Full Pipeline", fontsize=13, y=1.02)

    for ax, img, label, cmap in zip(axes, steps, STEP_LABELS, cmaps):
        if cmap:
            ax.imshow(img, cmap=cmap)
        elif img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(label, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


---
## Step 8 -- Agreed Parameters

Final values agreed after testing on the full dataset.
These are the exact values used in `main.py` and `app.py`.

| Step | Parameter | Value | Notes |
|------|-----------|-------|-------|
| Denoising | Method | NLM | Better than Median for edges |
| Denoising | h | 10 | |
| Denoising | templateWindowSize | 7 | |
| Denoising | searchWindowSize | 21 | |
| Scratch Detection | kernel_length | 23 | 4 directions: H,V,D1,D2 |
| Scratch Detection | bright_cutoff | 25 | exclude dark regions |
| Scratch Detection | adaptive_factor | 0.50 | threshold = max(25, p97 * 0.50) |
| Scratch Detection | min_scratch_len | 10 | |
| Scratch Detection | min_aspect_ratio | 2.0 | |
| Scratch Detection | density_threshold | 0.20 | area/bbox < 0.20 = line |
| Scratch Detection | max_mask_ratio | 0.08 | safety: reject if mask > 8% |
| Scratch Detection | dilation_iter | 1 | |
| Inpainting | Method | Navier-Stokes + blend | Better than TELEA for wide damage |
| Inpainting | radius | 7 | |
| Inpainting | blend_h | 5 | post-NLM to smooth seams |
| CLAHE | clip_limit | 2.0 | |
| CLAHE | tile_grid_size | (8, 8) | |
| Color | sepia_sat_min | 15 | |
| Color | sepia_sat_max | 60 | |
| Color | sepia_hue_min | 10 | |
| Color | sepia_hue_max | 30 | |
| Color (grayscale) | unsharp alpha | 2.5 | stronger than 1.5 to recover texture |
| Color (grayscale) | unsharp beta | -1.5 | |

> Update this table if you change any parameter and inform the whole team.


---
## Save Outputs


In [ ]:
os.makedirs("output", exist_ok=True)

for path, final in zip(IMAGE_PATHS, final_all):
    name     = os.path.splitext(os.path.basename(path))[0]
    out_path = os.path.join("output", f"restored_{name}.jpg")
    cv2.imwrite(out_path, final)
    print(f"Saved: {out_path}")

print(f"\nAll {len(final_all)} restored images saved to output/")
